# Task 2: DBSCAN Parameter Tuning

**Mục tiêu:** Tìm eps và min_samples tốt nhất cho DBSCAN clustering.

**Output:** `results/dbscan_recommendations.json`

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score
import os

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

## 1. Load Data

In [ ]:
df = pd.read_csv('../data/raw_tiktok_data.csv', encoding='utf-8-sig')
print(f"✅ Loaded {len(df)} videos")

## 2. Prepare Text Data

In [ ]:
# Filter videos có text
df_clean = df[df['text'].notna()].copy()
print(f"✅ Videos with text: {len(df_clean)}")

# Extract text + hashtags
hashtag_cols = [col for col in df.columns if col.startswith('hashtags/')]

def combine_text_hashtags(row):
    text = row['text']
    hashtags = [row[col] for col in hashtag_cols if pd.notna(row[col])]
    return f"{text} {' '.join(hashtags)}"

df_clean['combined_text'] = df_clean.apply(combine_text_hashtags, axis=1)

print(f"\nExample combined text:")
print(df_clean.iloc[0]['combined_text'][:200])

## 3. Generate Embeddings

**LƯU Ý:** Cần OPENAI_API_KEY để generate embeddings thật.

Nếu chưa có API key, có thể dùng **dummy embeddings** để test logic trước.

In [ ]:
# TODO: Chọn 1 trong 2 options

# OPTION A: Dummy embeddings (for testing)
USE_DUMMY_EMBEDDINGS = True  # Set False nếu có API key

if USE_DUMMY_EMBEDDINGS:
    print("⚠️  Using DUMMY embeddings (random vectors)")
    print("   For real results, set USE_DUMMY_EMBEDDINGS=False and provide OPENAI_API_KEY\n")
    
    embeddings = np.random.rand(len(df_clean), 1536)
    print(f"✅ Generated {embeddings.shape[0]} dummy embeddings")

else:
    # OPTION B: Real OpenAI embeddings
    from openai import OpenAI
    
    api_key = os.getenv('OPENAI_API_KEY')
    if not api_key:
        raise ValueError("OPENAI_API_KEY not found. Set: export OPENAI_API_KEY='sk-...'")
    
    client = OpenAI(api_key=api_key)
    
    # TODO: Embed texts
    texts = df_clean['combined_text'].tolist()
    
    print("Generating embeddings with OpenAI...")
    all_embeddings = []
    
    batch_size = 100
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        response = client.embeddings.create(
            model="text-embedding-3-small",
            input=batch
        )
        batch_embeddings = [item.embedding for item in response.data]
        all_embeddings.extend(batch_embeddings)
        print(f"  Processed {min(i+batch_size, len(texts))}/{len(texts)} texts")
    
    embeddings = np.array(all_embeddings)
    print(f"✅ Generated {embeddings.shape[0]} real embeddings")

## 4. DBSCAN Parameter Tuning

In [ ]:
# Test range
eps_values = [0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5]
min_samples_values = [2, 3, 5, 7, 10]

results = []

print("Testing DBSCAN parameters...\n")

for eps in eps_values:
    for min_samples in min_samples_values:
        # Run DBSCAN
        dbscan = DBSCAN(eps=eps, min_samples=min_samples, metric='cosine')
        labels = dbscan.fit_predict(embeddings)
        
        # Calculate metrics
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        n_noise = list(labels).count(-1)
        
        # Silhouette score (only if we have clusters)
        if n_clusters > 1:
            # Filter out noise for silhouette calculation
            mask = labels != -1
            if mask.sum() > 1:
                silhouette = silhouette_score(embeddings[mask], labels[mask], metric='cosine')
            else:
                silhouette = -1
        else:
            silhouette = -1
        
        # Average cluster size
        if n_clusters > 0:
            avg_cluster_size = (len(labels) - n_noise) / n_clusters
        else:
            avg_cluster_size = 0
        
        results.append({
            'eps': eps,
            'min_samples': min_samples,
            'n_clusters': n_clusters,
            'n_noise': n_noise,
            'silhouette_score': silhouette,
            'avg_cluster_size': avg_cluster_size
        })
        
        print(f"eps={eps:.2f}, min_samples={min_samples:2d} → "
              f"clusters={n_clusters:2d}, noise={n_noise:3d}, "
              f"silhouette={silhouette:.3f}, avg_size={avg_cluster_size:.1f}")

results_df = pd.DataFrame(results)
print(f"\n✅ Tested {len(results)} parameter combinations")

## 5. Visualize Results

In [ ]:
# Heatmap: Silhouette Score
pivot_silhouette = results_df.pivot(index='min_samples', columns='eps', values='silhouette_score')

plt.figure(figsize=(10, 6))
sns.heatmap(pivot_silhouette, annot=True, fmt='.3f', cmap='YlGnBu')
plt.title('Silhouette Score Heatmap')
plt.xlabel('eps')
plt.ylabel('min_samples')
plt.show()

In [ ]:
# Heatmap: Number of Clusters
pivot_clusters = results_df.pivot(index='min_samples', columns='eps', values='n_clusters')

plt.figure(figsize=(10, 6))
sns.heatmap(pivot_clusters, annot=True, fmt='.0f', cmap='viridis')
plt.title('Number of Clusters Heatmap')
plt.xlabel('eps')
plt.ylabel('min_samples')
plt.show()

In [ ]:
# Line chart: eps vs n_clusters (for different min_samples)
plt.figure(figsize=(12, 6))

for min_s in min_samples_values:
    subset = results_df[results_df['min_samples'] == min_s]
    plt.plot(subset['eps'], subset['n_clusters'], marker='o', label=f'min_samples={min_s}')

plt.xlabel('eps')
plt.ylabel('Number of Clusters')
plt.title('eps vs Number of Clusters')
plt.legend()
plt.grid(True)
plt.show()

## 6. Select Best Parameters

In [ ]:
# TODO: Chọn best params dựa vào:
# 1. Silhouette score cao nhất
# 2. Số clusters hợp lý (8-15 clusters)
# 3. Domain knowledge: 1 trend cần ít nhất 5 videos

# Filter: chỉ giữ configs có n_clusters >= 5 và <= 20
candidates = results_df[
    (results_df['n_clusters'] >= 5) & 
    (results_df['n_clusters'] <= 20)
]

# Sort by silhouette score
candidates_sorted = candidates.sort_values('silhouette_score', ascending=False)

print("Top 5 candidates:")
print(candidates_sorted.head())

# TODO: Chọn best
best = candidates_sorted.iloc[0]

print(f"\n✅ BEST PARAMETERS:")
print(f"   eps: {best['eps']}")
print(f"   min_samples: {best['min_samples']}")
print(f"   → {best['n_clusters']} clusters, silhouette={best['silhouette_score']:.3f}")

## 7. Save Results

In [ ]:
# TODO: Viết reasoning
reasoning = f"""Tested eps from 0.2-0.5 and min_samples 2-10.
Selected eps={best['eps']} and min_samples={int(best['min_samples'])} because:
- Highest silhouette score: {best['silhouette_score']:.3f}
- Reasonable number of clusters: {int(best['n_clusters'])}
- Average cluster size: {best['avg_cluster_size']:.1f} videos
- min_samples={int(best['min_samples'])} ensures each trend has enough videos
"""

output = {
    "best_params": {
        "eps": float(best['eps']),
        "min_samples": int(best['min_samples']),
        "metric": "cosine"
    },
    "reasoning": reasoning.strip(),
    "metrics": {
        "n_clusters": int(best['n_clusters']),
        "n_noise": int(best['n_noise']),
        "silhouette_score": float(best['silhouette_score']),
        "avg_cluster_size": float(best['avg_cluster_size'])
    },
    "all_results": results_df.to_dict('records')
}

# Save JSON
output_path = Path('../results/dbscan_recommendations.json')
output_path.parent.mkdir(exist_ok=True)

with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print(f"✅ Results saved to {output_path}")

In [ ]:
# Preview (without all_results)
preview = output.copy()
preview.pop('all_results')
print(json.dumps(preview, indent=2, ensure_ascii=False))

---

## ✅ DONE!

Kiểm tra file `results/dbscan_recommendations.json` đã được tạo chưa.